# §4.4 follow-up — which reference frame is the spectrum in?
### Using telluric lines as an absolute ruler

In [ ]:
import numpy as np, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from astropy.io import fits
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
import astropy.units as u
from PyAstronomy import pyasl
DATA = "../../../data"

def load(p):
    h = fits.open(p)[0]
    if len(fits.open(p)) > 1 and hasattr(fits.open(p)[1], "columns"):
        d = fits.open(p)[1].data
        w, f = np.asarray(d["wavelength"], float), np.asarray(d["norm_flux"], float)
    else:
        hd = h.header; n = hd["NAXIS1"]; cd = hd.get("CD1_1", hd.get("CDELT1"))
        w = hd["CRVAL1"] + cd*(np.arange(n) + 1 - hd.get("CRPIX1", 1))
        f = np.asarray(h.data, float)
    o = np.argsort(w); w, f = w[o], f[o]
    g = np.isfinite(w) & np.isfinite(f)
    return w[g], f[g]

S = {k: load(f"{DATA}/{v}") for k, v in
     dict(HE="he1158-2313_uncor.fits", HD140283="HD140283.fits", HD122563="HD122563.fits").items()}
for k, (w, f) in S.items(): print(f"{k:9s} {w[0]:.0f}-{w[-1]:.0f} A   n={len(w)}")

## 1. Are telluric lines even present?

In [ ]:
for k, (w, f) in S.items():
    for name, (a, b) in [("O2 A-band 7594-7660", (7594, 7660)),
                         ("O2 B-band 6867-6890", (6867, 6890)),
                         ("clean 7700-7750",     (7700, 7750))]:
        m = (w > a) & (w < b)
        print(f"{k:9s} {name:22s} median {np.nanmedian(f[m]):.3f}  min {np.nanmin(f[m]):.3f}")
    print()

## 2. The absolute anchor: HD 140283's own header

In [ ]:
h = fits.open(f"{DATA}/HD140283.fits")[0].header
dop = {k: h[k] for k in h if "DOPCOR" in k}
print("HD 140283 DOPCOR cards:", dop)
total = sum(float(str(v).split()[0]) for v in dop.values())
print(f"   net applied shift = {total:+.0f} km/s")

lco = EarthLocation.from_geodetic(lon=h["SITELONG"]*u.deg, lat=h["SITELAT"]*u.deg,
                                  height=h["SITEALT"]*u.m)
t = Time(f"{h['UT-DATE']}T{h['UT-START']}", scale="utc", location=lco)
sky = SkyCoord(ra=h["RA-D"]*u.deg, dec=h["DEC-D"]*u.deg)
hc = float(sky.radial_velocity_correction("heliocentric", obstime=t).to(u.km/u.s).value)
V_HELIO_140283 = -170.36                      # SIMBAD
v_geo_140283 = V_HELIO_140283 - hc
print(f"\n   UT {t.isot}   heliocentric correction {hc:+.3f} km/s")
print(f"   v_helio (SIMBAD) {V_HELIO_140283:+.2f}  ->  v_geo = {v_geo_140283:+.2f} km/s")
print(f"\n   The net DOPCOR ({total:+.0f}) equals the GEOCENTRIC velocity ({v_geo_140283:+.2f}).")
print("   So the reference spectra were built from geocentric MIKE output and shifted")
print("   straight to the star's rest frame. Its tellurics therefore sit at:")
V_TELL_140283 = -v_geo_140283
print(f"       v_tell(HD 140283) = {V_TELL_140283:+.2f} km/s")

## 3. Independent check that HD 140283 really is at rest

In [ ]:
LAB = {"H-alpha": 6562.797, "Mg I 5183.6": 5183.604, "Mg I 5172.7": 5172.684,
       "Ca I 4226.7": 4226.728, "Na D2": 5889.951, "Na D1": 5895.924}
w, f = S["HD140283"]; vs = []
for name, lab in LAB.items():
    m = (w > lab - 1.2) & (w < lab + 1.2)
    ww, ff = w[m], f[m]; i = np.argmin(ff)
    p = np.polyfit(ww[max(i-2, 0):i+3], ff[max(i-2, 0):i+3], 2)
    lam = -p[1]/(2*p[0]); v = 299792.458*(lam/lab - 1); vs.append(v)
    print(f"   {name:12s} core {lam:.3f} A   v = {v:+6.2f} km/s")
print(f"\n   median {np.median(vs):+.2f} km/s  (residual is ordinary convective blueshift)")
print("   at rest -> confirmed; a heliocentric-then-dopcor file would read -28.4")

## 4. Measure the telluric offset — and avoid the alias

In [ ]:
def ccf(a, b, lo, hi, vmin, vmax, step=0.5, pad=40):
    wa, fa = S[a]; wb, fb = S[b]
    ma = (wa > lo) & (wa < hi); mb = (wb > lo - pad) & (wb < hi + pad)
    r, c = pyasl.crosscorrRV(wa[ma], fa[ma] - np.nanmedian(fa[ma]),
                             wb[mb], fb[mb] - np.nanmedian(fb[mb]), vmin, vmax, step, skipedge=0)
    return r, c/np.max(c)

WIN = {"H2O 7160-7350 (irregular)": (7160, 7350), "H2O 8100-8350 (irregular)": (8100, 8350),
       "H2O 8900-9150 (irregular)": (8900, 9150), "O2 B-band 6865-6935": (6865, 6935),
       "O2 A-band 7594-7660 (COMB - aliases)": (7594, 7660)}
print(f"predicted HE - HD140283 :  heliocentric file {28.075 - V_TELL_140283:+.2f}"
      f" | geocentric file {0 - V_TELL_140283:+.2f}\n")
deltas = {}
for name, (lo, hi) in WIN.items():
    r, c = ccf("HE", "HD140283", lo, hi, -400, 200)
    d = float(r[np.argmax(c)]); deltas[name] = d
    print(f"   {name:38s} delta {d:+8.2f}  ->  v_tell(HE) {d + V_TELL_140283:+6.2f}")
clean = [v for k, v in deltas.items() if "COMB" not in k]
V_TELL_HE = float(np.mean(clean)) + V_TELL_140283
print(f"\n   excluding the aliasing A-band:  v_tell(HE) = {V_TELL_HE:+.2f} km/s")

## 5. The verdict

In [ ]:
helcorr_HE = 28.075   # 2013-01-09 07:23:55 UT, LCO (see part4-4_header_information)
print(f"   telluric velocity measured in he1158-2313_uncor.fits : {V_TELL_HE:+.2f} km/s")
print(f"   computed heliocentric correction for that night      : {helcorr_HE:+.2f} km/s")
print(f"   a geocentric file would give                         :   0.00 km/s")
print(f"\n   => the wavelength scale is ALREADY HELIOCENTRIC.")
print("      '_uncor' means 'not shifted to the star's rest frame', not 'geocentric'.")
print("      Adding the correction again would put us 28.3 km/s away from Gaia.")

fig, ax = plt.subplots(figsize=(11, 4))
for name, (lo, hi) in WIN.items():
    r, c = ccf("HE", "HD140283", lo, hi, -260, -20)
    ax.plot(r, c, lw=1.3, label=name)
for v, lab, col in [(28.075 - V_TELL_140283, "if heliocentric", "g"),
                    (0 - V_TELL_140283, "if geocentric", "r")]:
    ax.axvline(v, color=col, ls="--", lw=1.6, label=lab)
ax.set_xlabel("velocity shift, HE - HD 140283 (km/s)"); ax.set_ylabel("normalised CCF")
ax.legend(fontsize=8, frameon=False); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig("plots/telluric_frame_check.png", dpi=130)
plt.show()

## Why this matters beyond §4.4